In [ ]:
import numpy as np
try:
    import gymnasium as gym
    from gymnasium import spaces
except:
    import gym
    from gym import spaces

UP = 0
DOWN = 1
LEFT = 2
RIGHT = 3
PICK = 4
DROP = 5

ACTION_NAMES = {
    UP: "UP",
    DOWN: "DOWN", 
    LEFT: "LEFT",
    RIGHT: "RIGHT",
    PICK: "PICK",
    DROP: "DROP",
}

def clamp(v, lo, hi):
    if v < lo:
        return lo
    elif v > hi:
        return hi
    else:
        return v


In [ ]:
class WarehouseConfig:
    def __init__(self):
        self.grid_size = (6, 6)
        self.start_pos = None  # random placement
        self.pickup_pos = (0, 5)
        self.dropoff_pos = (5, 0)
        self.obstacles = set([
            (1, 1), (1, 2), (1, 3), 
            (3, 1), (3, 2), (3, 3),  
            (4, 4), (5, 4)           
        ])
        
        # Rewards
        self.step_penalty = -1.0
        self.obstacle_penalty = -20.0
        self.first_pick_reward = 25.0
        self.delivery_reward = 100.0
        
        # Rendering
        self.render_mode = "matplotlib"


class WarehouseBase(gym.Env):
    
    def __init__(self, config=None, seed=None):
        gym.Env.__init__(self)
        
        if config is None:
            config = WarehouseConfig()
        self.cfg = config
        
        self.rows, self.cols = self.cfg.grid_size
        self.obstacles = set(self.cfg.obstacles)
        self.pickup = self.cfg.pickup_pos
        self.dropoff = self.cfg.dropoff_pos

        self.observation_space = spaces.MultiDiscrete([self.rows, self.cols, 2])
        self.action_space = spaces.Discrete(6)

        self.rng = np.random.default_rng(seed)
        
        self.state = (0, 0)
        self.carrying = False
        self.picked_once = False
        self.delivered = False
        self.total_steps = 0
        self.total_reward = 0.0

    def get_valid_start_positions(self):
        valid_positions = []
        for r in range(self.rows):
            for c in range(self.cols):
                pos = (r, c)
                if (pos not in self.obstacles and 
                    pos != self.pickup and 
                    pos != self.dropoff):
                    valid_positions.append(pos)
        return valid_positions

    def move(self, action):
        r, c = self.state
        if action == UP:
            r -= 1
        elif action == DOWN:
            r += 1
        elif action == LEFT:
            c -= 1
        elif action == RIGHT:
            c += 1
        
        r = clamp(r, 0, self.rows - 1)
        c = clamp(c, 0, self.cols - 1)
        return (r, c)

    def apply_move_with_collisions(self, proposed):
        if isinstance(proposed, np.ndarray):
            proposed = tuple(proposed)
        if proposed in self.obstacles:
            return self.state, self.cfg.obstacle_penalty
        return (proposed, 0.0)

    def handle_pickup(self):
        if isinstance(self.state, tuple):
            robot_position = self.state
        else:
            robot_position = tuple(self.state)

        if robot_position != self.pickup:
            return 0.0
        if self.carrying:
            return 0.0
        self.carrying = True
        if self.picked_once is True:
            return 0.0
        else:
            self.picked_once = True
            return self.cfg.first_pick_reward

    def handle_dropoff(self):
        if isinstance(self.state, tuple):
            robot_position = self.state
        else:
            robot_position = tuple(self.state)
        
        if not self.carrying:
            return (0.0, False)
        
        if robot_position != self.dropoff:
            return (0.0, False)
        
        self.carrying = False
        self.delivered = True
        
        return (self.cfg.delivery_reward, True)

    def reset(self, seed=None, options=None):
        gym.Env.reset(self, seed=seed)
        
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        
        if self.cfg.start_pos is not None:
            self.state = self.cfg.start_pos
        else:
            valid_positions = self.get_valid_start_positions()
            if valid_positions:
                chosen_position = self.rng.choice(valid_positions)
                self.state = chosen_position
            else:
                self.state = (0, 0)
        
        self.carrying = False
        self.picked_once = False
        self.delivered = False
        self.total_steps = 0
        self.total_reward = 0.0
        
        observation = np.array([
            self.state[0],
            self.state[1],
            int(self.carrying)
        ], dtype=np.int64)
        
        environment_info = self.get_info()
        
        return observation, environment_info

    def get_info(self):
        return {
            "total_steps": self.total_steps,
            "total_reward": self.total_reward,
            "robot_pos": self.state,
            "carrying_item": self.carrying,
            "delivered": self.delivered
        }

    def step(self, action):
        self.total_steps += 1
        step_reward = self.cfg.step_penalty
        
        game_ended = False
        game_truncated = False
        
        if action in (UP, DOWN, LEFT, RIGHT):
            target_position = self.move(action)
            actual_position, collision_penalty = self.apply_move_with_collisions(target_position)
            step_reward += collision_penalty
            self.state = actual_position
            
        elif action == PICK:
            pickup_reward = self.handle_pickup()
            step_reward += pickup_reward
            
        elif action == DROP:
            drop_reward, is_completed = self.handle_dropoff()
            step_reward += drop_reward
            game_ended = is_completed
        
        self.total_reward += step_reward
        
        current_observation = np.array([
            self.state[0],
            self.state[1],
            int(self.carrying)
        ])
        
        current_info = self.get_info()
        
        return (
            current_observation,
            float(step_reward),
            bool(game_ended),
            bool(game_truncated),
            current_info
        )

    def close(self):
        pass


class WarehouseRobotDetEnv(WarehouseBase):
    def __init__(self, config=None, seed=None):
        WarehouseBase.__init__(self, config, seed)
        self.env_type = "Deterministic Warehouse Robot"


class MultiAgentWarehouseEnv(gym.Env):

   
    def __init__(self, base_env_class, config=None, num_agents=3, seed=None):
        gym.Env.__init__(self)
        
        self.num_agents = num_agents
        self.base_env_class = base_env_class
        
        if config is None:
            config = WarehouseConfig()
        
        self.cfg = config
        self.rows, self.cols = self.cfg.grid_size
        self.obstacles = set(self.cfg.obstacles)
        self.pickup = self.cfg.pickup_pos
        self.dropoff = self.cfg.dropoff_pos
        
        self.rng = np.random.default_rng(seed)
        
        self.agent_envs = []
        for i in range(num_agents):
            env = base_env_class(config=config, seed=seed)
            self.agent_envs.append(env)
        
        self.observation_space = spaces.Tuple([
            spaces.MultiDiscrete([self.rows, self.cols, 2]) 
            for _ in range(num_agents)
        ])
        
        self.action_space = spaces.Tuple([
            spaces.Discrete(6) for _ in range(num_agents)
        ])
        
        self.agent_states = [(0, 0) for _ in range(num_agents)]
        self.agent_carrying = [False for _ in range(num_agents)]
        self.agent_picked_once = [False for _ in range(num_agents)]
        self.agent_delivered = [False for _ in range(num_agents)]
        self.total_steps = 0
        
    def get_valid_start_positions(self):
        valid_positions = []
        for r in range(self.rows):
            for c in range(self.cols):
                pos = (r, c)
                if (pos not in self.obstacles and 
                    pos != self.pickup and 
                    pos != self.dropoff):
                    valid_positions.append(pos)
        return valid_positions
    
    def reset(self, seed=None, options=None):
        gym.Env.reset(self, seed=seed)
        
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        
        observations = []
        for i, env in enumerate(self.agent_envs):
            obs, info = env.reset(seed=seed)
            self.agent_states[i] = tuple(obs[:2])
            self.agent_carrying[i] = bool(obs[2])
            self.agent_picked_once[i] = False
            self.agent_delivered[i] = False
            observations.append(obs)
        
        self._resolve_start_position_conflicts()
        
        observations = []
        for i in range(self.num_agents):
            obs = np.array([
                self.agent_states[i][0],
                self.agent_states[i][1],
                int(self.agent_carrying[i])
            ], dtype=np.int64)
            observations.append(obs)
        
        self.total_steps = 0
        
        info = {
            "agent_positions": [self.agent_states[i] for i in range(self.num_agents)],
            "agent_carrying": [self.agent_carrying[i] for i in range(self.num_agents)],
            "agent_delivered": [self.agent_delivered[i] for i in range(self.num_agents)]
        }
        
        return tuple(observations), info
    
    def _resolve_start_position_conflicts(self):
        used_positions = set()
        valid_positions = self.get_valid_start_positions()
        
        for i in range(self.num_agents):
            current_pos = self.agent_states[i]
            if current_pos in used_positions:
                available = [p for p in valid_positions if p not in used_positions]
                if available:
                    new_pos = self.rng.choice(available)
                    self.agent_states[i] = new_pos
                    self.agent_envs[i].state = new_pos
            used_positions.add(self.agent_states[i])
    
    def _handle_agent_collisions(self, proposed_positions):
        final_positions = list(proposed_positions)
        position_counts = {}
        
        for i, pos in enumerate(final_positions):
            pos_tuple = tuple(pos)
            if pos_tuple not in position_counts:
                position_counts[pos_tuple] = []
            position_counts[pos_tuple].append(i)
        
        for pos_tuple, agent_indices in position_counts.items():
            if len(agent_indices) > 1:
                for agent_idx in agent_indices:
                    final_positions[agent_idx] = self.agent_states[agent_idx]
        
        return final_positions
    
    def step(self, actions):
        self.total_steps += 1
        
        individual_rewards = []
        proposed_positions = []
        
        for i, action in enumerate(actions):
            reward = self.cfg.step_penalty
            
            if action in (UP, DOWN, LEFT, RIGHT):
                r, c = self.agent_states[i]
                if action == UP:
                    r -= 1
                elif action == DOWN:
                    r += 1
                elif action == LEFT:
                    c -= 1
                elif action == RIGHT:
                    c += 1
                
                r = clamp(r, 0, self.rows - 1)
                c = clamp(c, 0, self.cols - 1)
                proposed_pos = (r, c)
                
                if proposed_pos in self.obstacles:
                    reward += self.cfg.obstacle_penalty
                    proposed_pos = self.agent_states[i]
                
                proposed_positions.append(proposed_pos)
                individual_rewards.append(reward)
            else:
                proposed_positions.append(self.agent_states[i])
                individual_rewards.append(self.cfg.step_penalty)
        
        final_positions = self._handle_agent_collisions(proposed_positions)
        
        for i, action in enumerate(actions):
            self.agent_states[i] = final_positions[i]
            self.agent_envs[i].state = final_positions[i]
            
            if action == PICK:
                reward = self._handle_pickup(i)
                individual_rewards[i] += reward
            elif action == DROP:
                reward, is_completed = self._handle_dropoff(i)
                individual_rewards[i] += reward
                if is_completed:
                    self.agent_delivered[i] = True
        
        all_delivered = all(self.agent_delivered)
        terminated = all_delivered
        
        observations = []
        for i in range(self.num_agents):
            obs = np.array([
                self.agent_states[i][0],
                self.agent_states[i][1],
                int(self.agent_carrying[i])
            ], dtype=np.int64)
            observations.append(obs)
        
        team_reward = sum(individual_rewards)
        
        info = {
            "individual_rewards": individual_rewards,
            "team_reward": team_reward,
            "agent_positions": [self.agent_states[i] for i in range(self.num_agents)],
            "agent_carrying": [self.agent_carrying[i] for i in range(self.num_agents)],
            "agent_delivered": [self.agent_delivered[i] for i in range(self.num_agents)],
            "total_steps": self.total_steps
        }
        
        return (
            tuple(observations),
            individual_rewards,
            terminated,
            False,
            info
        )
    
    def _handle_pickup(self, agent_idx):
        agent_pos = self.agent_states[agent_idx]
        
        if agent_pos != self.pickup:
            return 0.0
        if self.agent_carrying[agent_idx]:
            return 0.0
        
        self.agent_carrying[agent_idx] = True
        self.agent_envs[agent_idx].carrying = True
        
        if not self.agent_picked_once[agent_idx]:
            self.agent_picked_once[agent_idx] = True
            return self.cfg.first_pick_reward
        return 0.0
    
    def _handle_dropoff(self, agent_idx):
        agent_pos = self.agent_states[agent_idx]
        
        if not self.agent_carrying[agent_idx]:
            return (0.0, False)
        if agent_pos != self.dropoff:
            return (0.0, False)
        
        self.agent_carrying[agent_idx] = False
        self.agent_envs[agent_idx].carrying = False
        self.agent_delivered[agent_idx] = True
        
        return (self.cfg.delivery_reward, True)
    
    def render(self, mode="human"):
        print(f"Step: {self.total_steps}")
        print(f"Agent positions: {self.agent_states}")
        print(f"Agent carrying: {self.agent_carrying}")
        print(f"Agent delivered: {self.agent_delivered}")
        
        grid = [['.' for _ in range(self.cols)] for _ in range(self.rows)]
        
        for obs_pos in self.obstacles:
            grid[obs_pos[0]][obs_pos[1]] = '#'
        
        grid[self.pickup[0]][self.pickup[1]] = 'P'
        grid[self.dropoff[0]][self.dropoff[1]] = 'D'
        
        for i, pos in enumerate(self.agent_states):
            r, c = pos
            if grid[r][c] == 'P':
                grid[r][c] = 'P' + str(i+1) if not self.agent_carrying[i] else 'p' + str(i+1)
            elif grid[r][c] == 'D':
                grid[r][c] = 'D' + str(i+1) if not self.agent_carrying[i] else 'd' + str(i+1)
            else:
                grid[r][c] = str(i+1) if not self.agent_carrying[i] else chr(ord('a') + i)
        
        for row in grid:
            print(' '.join(row))
        print()
    
    def close(self):
        for env in self.agent_envs:
            env.close()
